# Lab 07 · Observability, tracing, and swappable agents

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'langsmith']):
    %pip install -q --upgrade "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Follow a decision back to its source

A trace connects a decision to earlier messages. These small Python records are teaching data; the main exercise adds structured logging, a registry, and optional LangSmith tracing.

In [ ]:
source = {"id": "m1", "sender": "guide", "text": "The park is closed today."}
decision = {"id": "m2", "sender": "planner", "text": "Visit the museum.", "cites": "m1"}
messages = {source["id"]: source, decision["id"]: decision}
print("Decision:", decision["text"])
print("Based on:", messages[decision["cites"]]["text"])

**Next:** Next, follow an architecture council's messages and evidence through structured logs.

## Follow a council's decisions
A final decision is easier to review when we can see who proposed it and which sources they used.

**So far:** Lab 06 debated this same support-search case with three architects and a vote. **New:** a smaller council as inspectable records — every message has an ID, every run shares a correlation ID, and reviewers live in a registry you can extend.

The council writes structured messages to a live log and a JSON log file. The registry is the `council` dictionary in Step 1; because the loop reads it, adding a rule-based reviewer takes one line, and the same steps run again unchanged.

The local log needs no extra key. An optional cell before Step 3 turns on LangSmith tracing, which records model calls separately from the lab's own IDs; use classroom data there, not confidential information.

**At a glance**

![Lab 07 at a glance: Minimalist (model agent) or Modernizer (model agent) or Rulebook (added in one line), then Schema check (structured message), then Write the log (JSON + trace_id), then Live log (same JSON lines) or council.log (one JSON per line) or LangSmith (trace tree (optional)), then Chair (majority or escalate), then Evidence trail (ids from council.log).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/07.svg)

## Think about the example

An agent review contains several messages about one decision. A trace ID groups the run; a message ID identifies one message. A source link records what a claim is based on.

**Concepts:** Correlation IDs across service boundaries; Structured logs and immutable event facts.

Use the scenario above to consider these questions before running the code:

1. What is the difference between a trace ID and a message ID?
2. Why does an event or agent message need both identity and provenance?

## The idea in context

Observability makes a workflow inspectable during and after a run. Each proposal is logged, and a registry lists reviewer settings.

Agent-run traces and microservice request traces share correlation principles but are different systems.

| Key term | Meaning |
| --- | --- |
| **Observability** | Seeing what a running system does from its outputs: logs, metrics, and traces. |
| **Trace** | A record of one run as a tree of steps, with each step's input, output, and timing; LangSmith shows traces of LLM applications. |
| **Log** | A time-stamped record of what happened; structured logs, such as JSON lines, can be searched and counted. |
| **Trace ID** | A label shared by one run's messages so they can be found and compared together; a message ID identifies a single message. |
| **Registry** | A lookup table of agents by name, so the system never hard-codes who takes part. |

## Pattern · Shared Memory

*Memory pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Registry & Adapter.*

| | |
| --- | --- |
| **Context** | Agents may work in separate contexts but still need a common source of truth for goals, progress, and handoffs. |
| **Problem** | Copying messages between agents duplicates context and creates inconsistent versions of the shared state. |
| **Solution** | Give the agents one shared memory, so that what one agent writes is immediately available to the others. |
| **Use when** | Agents share a project state; their own contexts should stay separate; a common task board is useful. |
| **Trade-off** | Reduces duplication, but concurrent updates need clear ownership to avoid overwrites and races. |
| **Flow** | Agent A → Shared memory → Agent B → Coordinated state |
| **Quality attributes** | Benefits: Interoperability, Reliability, Cost. Liabilities: Reliability, Security, Maintainability. |
| **In this lab** | The local log is a shared record for the chair and students. The reviewers do not automatically receive previous messages as context. This single-process example does not implement a concurrent or tamper-proof shared-memory service. |

## Main exercise at a glance

1. Prepare the case and reviewer settings
2. Set up a live log and a file
3. Collect and log proposals
4. Count valid proposals and log a decision
5. Follow the log and add a reviewer

## Step 1 · Prepare the case and reviewer settings
Each reviewer has a name, a kind, and a stance. Both start as model reviewers. This dictionary *is* the lab's registry: in Step 5 you add a rule reviewer to the same dictionary, and the loop in Step 3 reads either kind without changing. The cell also imports the modules the later steps need.

In [ ]:
import itertools, json, logging, sys
from collections import Counter
from datetime import datetime, timezone
from langsmith import trace

SOURCES = {"R2": "API composition fits small lookups by id; broad search can require large joins.",
           "R3": "CQRS supports rich queries but adds a stale read model.",
           "C1": "ShopEasy support searches orders and payments by name."}
CASE = {"name": "ShopEasy support", "cross_service_search": True}
OPTIONS = ["API composition", "CQRS"]
council = {
    "Minimalist": {"kind": "model", "stance": "Prefer the simplest design that works."},
    "Modernizer": {"kind": "model", "stance": "Prefer a read model for growing search traffic."},
}
ids = itertools.count()
print("Reviewers:", list(council))

## Step 2 · Set up a live log and a file
Each log line is a JSON message. The live handler displays it; the file handler saves the same line to `council.log`. Rerun this cell to start a new log.

In [ ]:
logger = logging.getLogger("council")
logger.setLevel(logging.INFO)
live = logging.StreamHandler(sys.stdout)
to_file = logging.FileHandler("council.log", mode="w")
formatter = logging.Formatter("%(message)s")
live.setFormatter(formatter)
to_file.setFormatter(formatter)
logger.handlers = [live, to_file]
logger.propagate = False
print("Log ready: council.log")

## Optional · Turn on LangSmith tracing
Two different things are called a trace in this lab. Every logged message carries a `trace_id` — a correlation label our own code chose, so one run's messages can be found together. A LangSmith trace is separate: the SDK's own record of the model calls, built while the code runs. Same correlation idea, two different systems.

Save `LANGSMITH_API_KEY` in Colab Secrets and run this cell before Step 3 to record model calls as well as messages. Tracing sends the case and model replies to LangSmith: use classroom data, not confidential information. Without a key nothing breaks — the local log keeps working.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"] = "true"
    print("LangSmith tracing is on for Steps 3–5.")
except Exception:
    print("No LANGSMITH_API_KEY found; local logging only.")

## Step 3 · Collect and log proposals
The loop reads each reviewer's settings. A model reviewer receives the case and sources; a rule reviewer uses a fixed condition. The `with trace` blocks record a parent run and one child run per reviewer when LangSmith is enabled.

Each proposal is logged with an ID and citations. Only a known option and known, nonempty citations pass the check. This checks the IDs, not whether a cited source supports the claim.

In [ ]:
accepted = []
with trace("council", inputs={"case": CASE}) as council_trace:
    for name, reviewer in council.items():
        with trace(name, inputs={"case": CASE}) as reviewer_trace:
            if reviewer["kind"] == "rule":
                option = "CQRS" if CASE["cross_service_search"] else "API composition"
                source = "R3" if CASE["cross_service_search"] else "R2"
                reply = {"option": option, "claim": "Review this read pattern and its cost.", "cites": [source, "C1"]}
            else:
                prompt = (f"You are {name}. {reviewer['stance']}\nSources: {json.dumps(SOURCES)}\n"
                          f"Case: {json.dumps(CASE)}\nReply with JSON only, no code fences: "
                          '{"option":"API composition or CQRS","claim":"one sentence","cites":["source ids"]}')
                reply = json.loads(ask(prompt))
            if not isinstance(reply, dict):
                reply = {}
            cites = reply.get("cites", [])
            valid = isinstance(cites, list) and bool(cites)
            if valid:
                for source_id in cites:
                    if not isinstance(source_id, str) or source_id not in SOURCES:
                        valid = False
            valid = valid and reply.get("option") in OPTIONS
            message = {"id": f"m{next(ids)}", "sender": name, "kind": "proposal" if valid else "rejected",
                       "option": str(reply.get("option")), "claim": str(reply.get("claim")),
                       "cites": cites, "trace_id": "support-review-1", "time": datetime.now(timezone.utc).isoformat()}
            logger.info(json.dumps(message))
            reviewer_trace.end(outputs={"reply": reply})
            if valid:
                accepted.append(message)
    council_trace.end(outputs={"accepted": accepted})

## Step 4 · Count valid proposals and log a decision
The chair counts the accepted proposals. It needs a strict majority of all registered reviewers; otherwise it escalates. The decision cites the IDs of the accepted proposals it used.

In [ ]:
votes = Counter()
for message in accepted:
    votes[message["option"]] += 1
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = 2 * count > len(council)
citations = []
for message in accepted:
    if not decided or message["option"] == option:
        citations.append(message["id"])
decision = {"id": f"m{next(ids)}", "sender": "Chair", "kind": "decision" if decided else "escalation",
            "option": option if decided else "no majority", "claim": f"{count} of {len(council)} reviewers",
            "cites": citations, "trace_id": "support-review-1", "time": datetime.now(timezone.utc).isoformat()}
logger.info(json.dumps(decision))
print("Decision:", decision)

## Step 5 · Follow the log and add a reviewer
Read the saved lines, find the proposal IDs cited by the decision, and print their sources. Then add a rule-based reviewer. Rerun Steps 3–5 to see its proposal included; the collection and voting loops stay the same.

The log is local. An ID identifies a message; the shared trace ID connects one run's messages. Rerun Steps 3–5 without touching Step 2 and the new lines append to the same `council.log`, so runs stay comparable in one file; rerun Step 2 only to start a fresh log.

In [ ]:
saved_messages = []
with open("council.log") as log_file:
    for line in log_file:
        saved_messages.append(json.loads(line))
print("Decision:", decision["id"], decision["option"])
for proposal_id in decision["cites"]:
    for message in saved_messages:
        if message["id"] == proposal_id:
            print("Proposal:", message["sender"], message["claim"])
            for source_id in message["cites"]:
                print("  Source:", source_id, SOURCES[source_id])
council["Rulebook"] = {"kind": "rule", "stance": "Use a simple search rule."}
print("Added Rulebook. Rerun Steps 3–5 to include it.")

## What you should see
The log shows two model proposals and a decision or escalation. Step 5 follows the decision's citations back to the source texts. Rerunning Steps 3–5 includes Rulebook as a third reviewer.

## Try it
1. Give a proposal an unknown source ID. Does it enter the vote?
2. Change one reviewer's stance and rerun from Step 3. How does the log help you compare runs?
3. Inspect `council.log`. Which ID identifies a message, and which trace ID connects the exercise?
4. Explain why a correlation ID across services is useful when one user request triggers several calls.

## Takeaway
Structured logs make it possible to follow a decision back to the proposals and sources. A registry lets the same loop include a model reviewer or a rule-based reviewer.